In [ ]:

#@title Install Qiskit
!pip install qiskit qiskit-aer
!pip install pylatexenc
!pip install matplotlib


In [ ]:
#@title Combined Alice-Bob Key Agreement for Trine1 and Trine2, with Encryption - Decryption
import random
import numpy as np
from collections import Counter
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister
from qiskit_aer import AerSimulator
from qiskit import transpile
import matplotlib.pyplot as plt

# ============================================================================
# PART 1: ENCDEC FUNCTIONS (Modified to work with Trine measurements)
# ============================================================================

def generate_random_array(size=1024, values=[0, 1, 2, 3]):
    """
    Generate an array of random numbers from specified values

    Args:
        size: Number of elements to generate
        values: List of possible values

    Returns:
        List of random numbers
    """
    return [random.choice(values) for _ in range(size)]

def transform_to_alice_final(pure_array):
    """
    Transform the pure array to alice_final array according to rules:
    - 0 → 0 (unchanged)
    - 1 → 1 (unchanged)
    - 2 → 0 (changed)
    - 3 → 1 (changed)

    Args:
        pure_array: Original array with values 0, 1, 2, 3

    Returns:
        Transformed array with values 0 and 1 only
    """
    mapping = {0: 0, 1: 1, 2: 0, 3: 1}
    return [mapping[value] for value in pure_array]

def transform_to_alice_pure_for_quantum(pure_array):
    """
    Transform pure array values to quantum states for Trine measurement:
    - 0 → |0⟩ state
    - 1 → |1⟩ state
    - 2 → |+⟩ state
    - 3 → |-⟩ state

    Args:
        pure_array: Original array with values 0, 1, 2, 3

    Returns:
        List of quantum state strings
    """
    mapping = {0: '0', 1: '1', 2: '+', 3: '-'}
    return [mapping[value] for value in pure_array]

def map_trine_outcome_to_bit(trine_outcome, trine_version = 1):
    """
    Map Trine measurement outcome to a bit value

    Args:
        trine_outcome: String 'E0', 'E1', or 'E2'

    Returns:
        Mapped bit (0 or 1)
    """

    if trine_version == 1:
      # Define mapping rules on Trine1
      if trine_outcome == "E0":
        return 0
      elif trine_outcome == "E2":
        return 1
      elif trine_outcome == "E1":
        raise ValueError(f"Invalid Trine outcome: {trine_string}. Expected 'E0', or 'E1'")
      else:
        raise ValueError(f"Invalid Trine outcome: {trine_string}. Expected 'E0', 'E1', or 'E2'")
    else: # trine_version == 2
      # Define mapping rules on Trine1
      if trine_outcome == "E0":
        return 1
      elif trine_outcome == "E1":
        return 0
      elif trine_outcome == "E2":
        return 1
      else:
        raise ValueError(f"Invalid Trine outcome: {trine_string}. Expected 'E0', 'E1', or 'E2'")



def analyze_array(arr, array_name="Array"):
    """
    Analyze the generated array

    Args:
        arr: List of numbers to analyze
        array_name: Name of the array for display
    """
    # Basic statistics
    print(f"\n{'='*50}")
    print(f"{array_name} Analysis:")
    print(f"{'='*50}")
    print(f"Array length: {len(arr)}")
    if len(arr) > 0:
        print(f"First 20 elements: {arr[:20]}")
        print(f"Last 20 elements: {arr[-20:]}")

    # Count frequencies
    frequencies = Counter(arr)
    print(f"\nFrequency of each value:")
    for num in sorted(frequencies.keys()):
        count = frequencies[num]
        percentage = (count / len(arr)) * 100 if len(arr) > 0 else 0
        print(f"  {num}: {count} times ({percentage:.2f}%)")

    # Verify all numbers are from the expected set
    unique_values = set(arr)
    print(f"\nUnique values in array: {sorted(unique_values)}")
    return unique_values

def Alice_Encoding(plaintext, alice_key, block_size = 256):
    """
    Encode a plaintext using Alice's key

    Args:
        plaintext: String of bits (e.g., "0011")
        alice_key: List of 1024 bits (0s and 1s)

    Returns:
        ciphertext: String of 1024 bits
    """
    # Convert alice_key list to string for easier manipulation
    alice_key_str = ''.join(str(bit) for bit in alice_key)

    # Generate extended_plaintext by copying each bit 'lock_size' times
    extended_plaintext = ""
    for bit in plaintext:
        extended_plaintext += bit * block_size

    # XOR extended_plaintext with alice_key
    ciphertext = ""
    for i in range(len(plaintext)*block_size):
        if extended_plaintext[i] == alice_key_str[i]:
            ciphertext += "0"
        else:
            ciphertext += "1"

    return ciphertext

def Bob_Decoding(ciphertext, bob_key, block_size = 256):
    """
    Decode a ciphertext using Bob's key with majority voting

    Args:
        ciphertext: String of 1024 bits
        bob_key: List of 1024 bits (0s and 1s)

    Returns:
        plaintext_result: String of 4 bits
    """
    # Convert bob_key list to string for easier manipulation
    bob_key_str = ''.join(str(bit) for bit in bob_key)

    # XOR ciphertext with bob_key
    xor_result = ""
    for i in range(len(ciphertext)):
        if ciphertext[i] == bob_key_str[i]:
            xor_result += "0"
        else:
            xor_result += "1"

    # For each 256-bit block, determine majority bit
    plaintext_result = ""
    for i in range(0, len(ciphertext), block_size):
        block = xor_result[i:i+block_size]
        zeros = block.count("0")
        ones = block.count("1")

        if zeros > ones:
            plaintext_result += "0"
        else:
            plaintext_result += "1"

    return plaintext_result

# ============================================================================
# PART 2: TRINE QUANTUM MEASUREMENT FUNCTIONS
# ============================================================================

# This function is the most important in the entire code,
# as it shows the TWO trine implementations
def create_trine_circuit(quantum_state, trine_version = 1):
    """
    Create a quantum circuit for Trine measurement on a specific state

    Args:
        quantum_state: String '0', '1', '+', or 'i'

    Returns:
        QuantumCircuit configured for Trine measurement
    """


    # Create quantum and classical registers
    qr = QuantumRegister(1, 'q')
    qr2 = QuantumRegister(1, 'q2')
    cr = ClassicalRegister(2, 'c')
    qc = QuantumCircuit(qr, qr2, cr)

    # Initialize the qubit in the desired state
    if quantum_state == '0':
        # |0> state (already initialized)
        pass
    elif quantum_state == '1':
        # |1> state
        qc.x(0)
    elif quantum_state == '+':
        # |+> state
        qc.h(0)
    elif quantum_state == '-':
        # |i> state = (|0> + i|1>)/√2
        qc.x(0)
        qc.h(0)
    else:
        raise ValueError(f"State must be '0', '1', '+', or '-', got {quantum_state}")

    # Implement Trine measurement using Naimark dilation
    if trine_version == 1:
        qc.h(qr2[0])
        qc.cx(qr[0], qr2[0])
        qc.rz(2*np.pi/3, qr[0])
        qc.cx(qr[0], qr2[0])
        qc.h(qr2[0])
    else:
        print("*********************")
        print("*********************")
        print("*********************")
        qc.h(1)              # Put ancilla in |+>
        qc.cry(1.231, 0, 1)  # Rotate system qubit
        qc.h(0)          # More entanglement
        qc.cx(0, 1)          # More entanglement
        qc.h(0)          # More entanglement


    # Measure both qubits
    qc.measure(qr[0], cr[0])
    qc.measure(qr2[0], cr[1])

    return qc

def run_trine_measurement(quantum_state, shots=1, trine_version = 1):
    """
    Run a single Trine measurement on a quantum state

    Args:
        quantum_state: String '0', '1', '+', or 'i'
        shots: Number of shots (should be 1 for single measurement)

    Returns:
        Trine outcome string ('E0', 'E1', or 'E2')
    """

    # Create the circuit
    qc = create_trine_circuit(quantum_state, trine_version)

    # Create simulator
    simulator = AerSimulator()

    # Compile the circuit for the simulator
    compiled_circuit = transpile(qc, simulator)

    # Run the job
    # job = simulator.run(compiled_circuit, shots=shots)
    job = simulator.run(compiled_circuit, shots=1) # Allow onlynone shot
    result = job.result()
    counts = result.get_counts()

    # Get the most likely outcome (for shots=1, this is the only outcome)
    # Map the 2-bit outcomes to Trine measurement results
    # 00 -> E0, 10 -> E1, 01 -> E2
    if '00' in counts:
        return 'E0'
    elif '10' in counts:
        return 'E1'
    elif '01' in counts:
        return 'E2'
    else:
        # Fallback in case of '11' outcome (shouldn't happen theoretically)
        return random.choice(['E0', 'E1', 'E2'])

def perform_bob_measurements(quantum_state_array, pure_array, trine_version = 1):
    """
    Perform Trine measurements for all quantum states and map to Bob's key

    Args:
        quantum_state_array: List of quantum state strings
        pure_array: Original pure array values

    Returns:
        bob_key: List of 1024 bits (Bob's decryption key)
        trine_outcomes: List of Trine outcomes for analysis
    """

    bob_key = []
    trine_outcomes = []

    print("\nPerforming Trine measurements for ", len(pure_array), " quantum states...")
    print("Progress: ", end="")

    for i, (quantum_state, pure_val) in enumerate(zip(quantum_state_array, pure_array)):
        if i % 100 == 0 and i > 0:
            print(f"{i} ", end="", flush=True)

        # Run Trine measurement
        trine_outcome = run_trine_measurement(quantum_state, 1, trine_version) # one shot
        trine_outcomes.append(trine_outcome)

        # Map to bit
        bit = map_trine_outcome_to_bit(trine_outcome, trine_version)
        bob_key.append(bit)

    print("1024 done!")
    return bob_key, trine_outcomes

def analyze_trine_outcomes(trine_outcomes, pure_array):
    """
    Analyze the distribution of Trine outcomes

    Args:
        trine_outcomes: List of Trine outcome strings
        pure_array: Original pure array values
    """

    # Overall statistics
    outcome_counts = Counter(trine_outcomes)
    total = len(trine_outcomes)

    print("\nOverall Trine outcome distribution:")
    for outcome in ['E0', 'E1', 'E2']:
        count = outcome_counts.get(outcome, 0)
        percentage = (count / total) * 100
        print(f"  {outcome}: {count} ({percentage:.2f}%)")

    # Conditional statistics based on pure value and quantum state
    print("\nConditional outcome distribution by pure value:")

    # Map pure values to quantum states for reference
    pure_to_quantum = {0: '0', 1: '1', 2: '+', 3: 'i'}

    for pure_val in sorted(set(pure_array)):
        indices = [i for i, p in enumerate(pure_array) if p == pure_val]
        outcomes_for_pure = [trine_outcomes[i] for i in indices]

        if outcomes_for_pure:
            counts = Counter(outcomes_for_pure)
            total_pure = len(outcomes_for_pure)

            print(f"\n  Pure = {pure_val} (Quantum state |{pure_to_quantum[pure_val]}⟩):")
            for outcome in ['E0', 'E1', 'E2']:
                count = counts.get(outcome, 0)
                percentage = (count / total_pure) * 100
                print(f"    {outcome}: {count}/{total_pure} ({percentage:.2f}%)")

# ============================================================================
# PART 3: MAIN INTEGRATED WORKFLOW
# ============================================================================

def run_trine(plaintext, block_size, trine_version = 1):
    '''
    Simulate running Trine1 protocol

    Args:
        block_size: Block size for encryption
        plaintext: Binary plaintext string
    '''

    print("#########################. POINT 1 #####################")
    print(trine_version)
    print("#########################")


    key_length = len(plaintext)*block_size

    print("'" * 80)
    print("INTEGRATED ALICE-BOB ENCRYPTION WITH TRINE QUANTUM MEASUREMENTS")
    print("=" * 80)

    # Step 1: Generate Alice's pure array
    print("\n[STEP 1] Generating Alice's pure array...")
    pure_array = generate_random_array(key_length, values=[0, 1, 2, 3])
    analyze_array(pure_array, "Pure Array (0, 1, 2, 3)")

    # Step 2: Transform to Alice's encryption key
    print("\n[STEP 2] Transforming to Alice's encryption key...")
    alice_encryption_key = transform_to_alice_final(pure_array)
    analyze_array(alice_encryption_key, "Alice Encryption Key (0, 1 only)")

    # Step 3: Transform pure array to quantum states for Bob's measurements
    print("\n[STEP 3] Preparing quantum states for Bob's measurements...")
    quantum_state_array = transform_to_alice_pure_for_quantum(pure_array)

    # Show mapping for first 20 elements
    print("\nMapping pure values to quantum states (first 20):")
    print("Index | Pure | Quantum State")
    print("-" * 35)
    for i in range(20):
        print(f"{i:5d} | {pure_array[i]:4d} | {quantum_state_array[i]:12s}")

    # Step 4: Bob performs Trine measurements
    print("\n[STEP 4] Bob performs Trine measurements...")
    bob_decryption_key, trine_outcomes = perform_bob_measurements(quantum_state_array, pure_array, trine_version)

    # Step 5: Analyze Trine outcomes
    analyze_trine_outcomes(trine_outcomes, pure_array)

    # Step 6: Analyze Bob's decryption key
    print("\n[STEP 6] Bob's decryption key analysis...")
    analyze_array(bob_decryption_key, "Bob Decryption Key (0, 1 only)")

    # Step 7: Compare Alice and Bob keys
    print("\n[STEP 7] Comparing Alice and Bob keys...")
    key_matches = sum(1 for a, b in zip(alice_encryption_key, bob_decryption_key) if a == b)
    key_match_percentage = (key_matches / len(alice_encryption_key)) * 100
    print(f"\nKey agreement rate: {key_matches}/{len(alice_encryption_key)} = {key_match_percentage:.2f}%")

    # Show first 20 bits of both keys
    print("\nFirst 20 bits of both keys:")
    print("Index | Alice Key | Bob Key | Match? | Pure | Bob-Trine")
    print("-" * 40)
    for i in range(20):
        alice_bit = alice_encryption_key[i]
        bob_bit = bob_decryption_key[i]
        match = "✓" if alice_bit == bob_bit else "✗"
        pure_array_bit = pure_array[i]
        trine_outcomes_bit = trine_outcomes[i]
        print(f"{i:5d} | {alice_bit:9d} | {bob_bit:7d} | {match:5s}| {pure_array_bit:7d}| {trine_outcomes_bit:7s}")


    print(f"\nUsing Alice Encryption Key and Bob Decryption Key")
    print(f"Alice key (first 20 bits): {alice_encryption_key[:20]}")
    print(f"Bob key (first 20 bits): {bob_decryption_key[:20]}")

    # Summary
    print("\n" + "=" * 80)
    print("EXPERIMENT SUMMARY")
    print("=" * 80)
    print(f"Total pure values: ", key_length)
    print(f"Alice-Bob key agreement rate: {key_match_percentage:.2f}%")


    print("\nMapping from pure values to quantum states:")
    print("  Pure 0 → |0⟩")
    print("  Pure 1 → |1⟩")
    print("  Pure 2 → |+⟩")
    print("  Pure 3 → |-⟩")

    print("\nMapping from Trine outcomes to bits:")
    print("  E0 → 0")
    print("  E2 → 1")

    # Step 8: Encryption/Decryption with majority voting
    print("\n" + "=" * 80)
    print("[STEP 8] ENCODING/DECODING WITH MAJORITY VOTING")
    print("=" * 80)

    print(f"\nOriginal plaintext: {plaintext}")
    print(f"\nBlock size - copies of each bit: {block_size}")

    print(f"\nUsing Alice Encryption Key and Bob Decryption Key")
    print(f"Alice key (first 20 bits): {alice_encryption_key[:20]}")
    print(f"Bob key (first 20 bits): {bob_decryption_key[:20]}")

    # Call Alice_Encoding to get ciphertext
    ciphertext = Alice_Encoding(plaintext, alice_encryption_key, block_size)
    print(f"\nCiphertext (first 20 bits): {ciphertext[:20]}...")
    print(f"Ciphertext (last 20 bits): ...{ciphertext[-20:]}")

    # Call Bob_Decoding to recover plaintext
    plaintext_result = Bob_Decoding(ciphertext, bob_decryption_key, block_size)
    print(f"\nDecoded plaintext result: {plaintext_result}")

    # Check if they match
    if plaintext == plaintext_result:
        print("✓ SUCCESS: Original plaintext and decoded plaintext match!")
    else:
        print("✗ ERROR: Original plaintext and decoded plaintext do not match!")

    # Detailed majority voting analysis
    print(f"\n{'='*50}")
    print("DETAILED MAJORITY VOTING ANALYSIS")
    print(f"{'='*50}")

    # For each of the 4 blocks (block_size bits each), analyze the XOR result patterns
    for block_num in range(len(plaintext)):
        start_idx = block_num * block_size
        end_idx = start_idx + block_size

        # Get the XOR result for this block after Bob's decoding step
        bob_xor_result = ""
        for i in range(start_idx, end_idx):
            if ciphertext[i] == str(bob_decryption_key[i]):
                bob_xor_result += "0"
            else:
                bob_xor_result += "1"

        zeros = bob_xor_result.count("0")
        ones = bob_xor_result.count("1")
        majority = "0" if zeros > ones else "1"
        expected_bit = plaintext[block_num]

        # Calculate how many bits in this block match between Alice and Bob
        block_key_matches = sum(1 for j in range(start_idx, end_idx)
                               if alice_encryption_key[j] == bob_decryption_key[j])
        block_key_match_percentage = (block_key_matches / block_size) * 100

        print(f"\nBlock {block_num+1} (bits {start_idx}-{end_idx-1}):")
        print(f"  Expected bit from plaintext: {expected_bit}")
        print(f"  Key agreement in this block: {block_key_matches}/block_size ({block_key_match_percentage:.2f}%)")
        print(f"  Zeros in XOR result: {zeros}")
        print(f"  Ones in XOR result: {ones}")
        print(f"  Majority bit: {majority}")
        print(f"  Match: {'✓' if majority == expected_bit else '✗'}")

    # Summary
    print("\n" + "=" * 80)
    print("EXPERIMENT SUMMARY")
    print("=" * 80)
    print(f"Total pure values: ", key_length)
    print(f"Alice-Bob key agreement rate: {key_match_percentage:.2f}%")
    print(f"Original plaintext: {plaintext}")
    print(f"Decoded plaintext: {plaintext_result}")
    print(f"Encryption/Decryption success: {'✓' if plaintext == plaintext_result else '✗'}")



'''
if __name__ == "__main__":

    # Run the main integrated workflow
    main()
'''


In [ ]:
#@title Trine 1 and Trine 2 Menu System
import os
import sys

def clear_screen():
    """Clear the console screen"""
    os.system('cls' if os.name == 'nt' else 'clear')

def display_menu():
    """Display the main menu options"""
    print("\n" + "="*60)
    print("         QUANTUM ENCRYPTION MENU SYSTEM")
    print("="*60)
    print("1. Run Trine1 Protocol")
    print("2. Run Trine2 Protocol")
    print("3. Exit")
    print("="*60)

def get_block_size(default=256):
    """
    Get block size from user with default value

    Args:
        default: Default block size (256)

    Returns:
        int: Block size entered by user
    """
    while True:
        try:
            user_input = input(f"\nEnter block size [default: {default}]: ").strip()

            # If user presses Enter with no input, use default
            if user_input == "":
                print(f"Using default block size: {default}")
                return default

            # Convert to integer and validate
            block_size = int(user_input)
            if block_size <= 0:
                print("Error: Block size must be positive. Please try again.")
                continue
            if block_size % 8 != 0:
                print("Warning: Block size not a multiple of 8. Continue anyway? (y/n): ", end="")
                if input().lower() != 'y':
                    continue

            return block_size

        except ValueError:
            print("Error: Please enter a valid integer.")

def get_plaintext(default="0011"):
    """
    Get plaintext in binary from user with default value

    Args:
        default: Default plaintext (0011)

    Returns:
        str: Binary plaintext entered by user
    """
    while True:
        user_input = input(f"\nEnter plaintext in binary [default: {default}]: ").strip()

        # If user presses Enter with no input, use default
        if user_input == "":
            print(f"Using default plaintext: {default}")
            return default

        # Validate binary string
        if all(c in '01' for c in user_input):
            return user_input
        else:
            print("Error: Plaintext must contain only 0s and 1s. Please try again.")

def main():
    """Main menu loop"""
    # Default values
    default_block_size = 256
    default_plaintext = "0011"


    clear_screen()
    display_menu()

    choice = input("\nEnter your choice (1-3): ").strip()

    if choice == '1':
        print("\n--- Trine1 Configuration ---")
        plaintext = get_plaintext(default_plaintext)
        block_size = get_block_size(default_block_size)

        print("\n" + "="*60)
        print("CONFIRMATION")
        print("="*60)
        print(f"Protocol: Trine1")
        print(f"Block size: {block_size}")
        print(f"Plaintext: {plaintext}")

        confirm = input("\nProceed? (y/n): ").strip().lower()
        if confirm == 'y':
            run_trine(plaintext, block_size, 1) # trine_version is 1
        else:
            print("\nOperation cancelled.")

        input("\nPress Enter to continue...")

    elif choice == '2':
        print("\n--- Trine2 Configuration ---")
        plaintext = get_plaintext(default_plaintext)
        block_size = get_block_size(default_block_size)

        print("\n" + "="*60)
        print("CONFIRMATION")
        print("="*60)
        print(f"Protocol: Trine2")
        print(f"Block size: {block_size}")
        print(f"Plaintext: {plaintext}")

        confirm = input("\nProceed? (y/n): ").strip().lower()
        if confirm == 'y':
            run_trine(plaintext, block_size, 2) # trine_version is 2
        else:
            print("\nOperation cancelled.")


    elif choice == '3':
        print("\nExiting program. Goodbye!")

    else:
        print("\nInvalid choice. Please enter 1, 2, or 3.")
        input("\nPress Enter to continue...")


if __name__ == "__main__":
    # Set random seed for reproducibility
    random.seed(42)
    np.random.seed(42)

    main()